In [8]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata

# === Date & Paths ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

paths = {
    "spreads": f"../data/novig-odds/novig_spreads_{today_str}.csv",
    "pitching_stats": f"../data/pitching_stats/pitching_stats_{today_str}.csv",
    "standings": f"../data/standings/standings_{today_str}.csv",
    "team_batting": f"../data/team_batting/team_batting_{today_str}.csv",
    "team_pitching": f"../data/team_pitching/team_pitching_{today_str}.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitching_stats_df = pd.read_csv(paths["pitching_stats"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
teamp_df = pd.read_csv(paths["team_pitching"])

# === Normalize ===
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Preprocess Spreads ===
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize team names ===
pitching_stats_df['Home Team Clean'] = pitching_stats_df['Home Team'].apply(normalize_str)
pitching_stats_df['Away Team Clean'] = pitching_stats_df['Away Team'].apply(normalize_str)

# === Map team abbreviations to full names ===
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "KAN": "Kansas City Royals", "AZ": "Arizona Diamondbacks", "ARI": "Arizona Diamondbacks",
    "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres", "MIN": "Minnesota Twins",
    "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# Derive home and away teams
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_team_full'] = spreads_df['home_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['away_team_full'] = spreads_df['away_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['home_team_clean'] = spreads_df['home_team_full'].apply(normalize_str)
spreads_df['away_team_clean'] = spreads_df['away_team_full'].apply(normalize_str)

# === Assign Pitchers and Stats ===
output_rows = []

for _, row in spreads_df.iterrows():
    home = row['home_team_clean']
    away = row['away_team_clean']

    ps_row = pitching_stats_df[
        (pitching_stats_df['Home Team Clean'] == home) &
        (pitching_stats_df['Away Team Clean'] == away)
    ]

    if ps_row.empty:
        ps_row = pitching_stats_df[
            (pitching_stats_df['Home Team Clean'] == away) &
            (pitching_stats_df['Away Team Clean'] == home)
        ]
        flipped = not ps_row.empty
    else:
        flipped = False

    if not ps_row.empty:
        ps_row = ps_row.iloc[0]
        if flipped:
            home_pitcher, away_pitcher = ps_row['Away Starter'], ps_row['Home Starter']
            home_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }
            away_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
        else:
            home_pitcher, away_pitcher = ps_row['Home Starter'], ps_row['Away Starter']
            home_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
            away_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }

        if row['home_favorite'] == 1:
            fav_pitcher, dog_pitcher = home_pitcher, away_pitcher
            fav_stats, dog_stats = home_stats, away_stats
        else:
            fav_pitcher, dog_pitcher = away_pitcher, home_pitcher
            fav_stats, dog_stats = away_stats, home_stats
    else:
        fav_pitcher = dog_pitcher = "Unknown"
        fav_stats = dog_stats = {k: None for k in ['era', 'whip', 'so', 'ip', 'avg']}

    row = row.copy()
    row['fav_pitcher'] = fav_pitcher
    row['dog_pitcher'] = dog_pitcher
    for k in fav_stats:
        row[f'fav_pitcher_{k}'] = fav_stats[k]
        row[f'dog_pitcher_{k}'] = dog_stats[k]

    # === New SO/IP Stat ===
    row['fav_pitcher_so_in'] = (
        fav_stats['so'] / fav_stats['ip'] if fav_stats['so'] is not None and fav_stats['ip'] else None
    )
    row['dog_pitcher_so_in'] = (
        dog_stats['so'] / dog_stats['ip'] if dog_stats['so'] is not None and dog_stats['ip'] else None
    )

    output_rows.append(row)

merged_df = pd.DataFrame(output_rows)

# === Normalize and Join Team Stats ===
for df in [standings_df, batting_df, teamp_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Standings
merged_df['fav_win_pct'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
merged_df['dog_win_pct'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))

# Batting/ERA
merged_df['fav_ba'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['dog_ba'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['fav_era_p'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))
merged_df['dog_era_p'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))

# Additional standings columns
standings_columns = ['W','L','Strk','R','RA','SOS','SRS','pythWL','Luck','last10','last20','last30']
for col in standings_columns:
    merged_df[f'fav_{col}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))
    merged_df[f'dog_{col}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))

# === Convert win/loss streaks to signed integers ===
def convert_streak(s):
    if isinstance(s, str) and len(s) > 1:
        direction = 1 if s[0].upper() == 'W' else -1 if s[0].upper() == 'L' else 0
        try:
            count = int(s[1:])
            return direction * count
        except ValueError:
            return None
    return None

merged_df['fav_Strk'] = merged_df['fav_Strk'].apply(convert_streak)
merged_df['dog_Strk'] = merged_df['dog_Strk'].apply(convert_streak)

# === Convert pythWL to win percentages ===
def wl_str_to_pct(wl_str):
    if isinstance(wl_str, str) and '-' in wl_str:
        try:
            wins, losses = map(int, wl_str.split('-'))
            total = wins + losses
            return wins / total if total > 0 else None
        except:
            return None
    return None

merged_df['fav_pythWL_pct'] = merged_df['fav_pythWL'].apply(wl_str_to_pct)
merged_df['dog_pythWL_pct'] = merged_df['dog_pythWL'].apply(wl_str_to_pct)

# Optional: drop the original 'W-L' strings
merged_df.drop(columns=['fav_pythWL', 'dog_pythWL'], inplace=True)

# === Convert last10/20/30 to win percentages ===
def wl_to_pct(wl_str):
    if isinstance(wl_str, str) and '-' in wl_str:
        try:
            wins, losses = map(int, wl_str.split('-'))
            total = wins + losses
            return wins / total if total > 0 else None
        except:
            return None
    return None

for span in ['last10', 'last20', 'last30']:
    merged_df[f'fav_{span}_pct'] = merged_df[f'fav_{span}'].apply(wl_to_pct)
    merged_df[f'dog_{span}_pct'] = merged_df[f'dog_{span}'].apply(wl_to_pct)

# Optional: drop original string columns
merged_df.drop(columns=['fav_last10', 'dog_last10', 'fav_last20', 'dog_last20', 'fav_last30', 'dog_last30'], inplace=True)

# Additional batting columns
batting_columns = ['R/G','PA','AB','R','H','2B','3B','HR','RBI','SB','BB','SO','BA','OBP','SLG','OPS','OPS+']
for col in batting_columns:
    col_renamed = f'{col}_b' if col in ['R','H','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))

# === Convert counting stats to rates per AB (percentages) ===
def safe_divide(numerator, denominator):
    try:
        num = float(numerator)
        denom = float(denominator)
        return num / denom if denom > 0 else None
    except (ValueError, TypeError):
        return None

conversion_pairs = [
    ('fav_2B', 'fav_AB', 'fav_2B_pct'),
    ('dog_2B', 'dog_AB', 'dog_2B_pct'),
    ('fav_3B', 'fav_AB', 'fav_3B_pct'),
    ('dog_3B', 'dog_AB', 'dog_3B_pct'),
    ('fav_HR', 'fav_AB', 'fav_HR_pct'),
    ('dog_HR', 'dog_AB', 'dog_HR_pct'),
    ('fav_RBI', 'fav_AB', 'fav_RBI_pct'),
    ('dog_RBI', 'dog_AB', 'dog_RBI_pct'),
]

for stat_col, ab_col, pct_col in conversion_pairs:
    merged_df[pct_col] = merged_df.apply(lambda row: safe_divide(row[stat_col], row[ab_col]), axis=1)


# Additional pitching columns
pitching_columns = ['RA/G','ERA','SV','H','R','ER','HR','BB','SO','ERA+','FIP','WHIP','H9']
for col in pitching_columns:
    col_renamed = f'{col}_p' if col in ['H','R','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))

# === Final Ordering ===
first_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so', 'dog_pitcher_so',
    'fav_pitcher_ip', 'dog_pitcher_ip',
    'fav_pitcher_avg', 'dog_pitcher_avg',
    'fav_win_pct', 'dog_win_pct', 'fav_ba', 'dog_ba', 'fav_era_p', 'dog_era_p'
]
remaining_cols = [col for col in merged_df.columns if col not in first_cols and col not in ['market_timestamp', 'home_team_full', 'away_team_full', 'home_team_clean', 'away_team_clean']]
merged_df = merged_df[first_cols + remaining_cols]

# === Output ===
print("\u2705 Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(merged_df)

# === Save CSV ===
output_path = f"../training-data/training-set/training_set_{today_str}.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged_df.to_csv(output_path, index=False)
print(f"\u2705 Enriched training set saved to {output_path}")

✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so,dog_pitcher_so,fav_pitcher_ip,dog_pitcher_ip,fav_pitcher_avg,dog_pitcher_avg,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era_p,dog_era_p,fav_pitcher_so_in,dog_pitcher_so_in,fav_W,dog_W,fav_L,dog_L,fav_Strk,dog_Strk,fav_R,dog_R,fav_RA,dog_RA,fav_SOS,dog_SOS,fav_SRS,dog_SRS,fav_Luck,dog_Luck,fav_pythWL_pct,dog_pythWL_pct,fav_last10_pct,dog_last10_pct,fav_last20_pct,dog_last20_pct,fav_last30_pct,dog_last30_pct,fav_R/G,dog_R/G,fav_PA,dog_PA,fav_AB,dog_AB,fav_R_b,dog_R_b,fav_H_b,dog_H_b,fav_2B,dog_2B,fav_3B,dog_3B,fav_HR,dog_HR,fav_RBI,dog_RBI,fav_SB,dog_SB,fav_BB_b,dog_BB_b,fav_SO_b,dog_SO_b,fav_BA,dog_BA,fav_OBP,dog_OBP,fav_SLG,dog_SLG,fav_OPS,dog_OPS,fav_OPS+,dog_OPS+,fav_2B_pct,dog_2B_pct,fav_3B_pct,dog_3B_pct,fav_HR_pct,dog_HR_pct,fav_RBI_pct,dog_RBI_pct,fav_RA/G,dog_RA/G,fav_ERA,dog_ERA,fav_SV,dog_SV,fav_H_p,dog_H_p,fav_R_p,dog_R_p,fav_ER,dog_ER,fav_BB_p,dog_BB_p,fav_SO_p,dog_SO_p,fav_ERA+,dog_ERA+,fav_FIP,dog_FIP,fav_WHIP,dog_WHIP,fav_H9,dog_H9
0,2025-06-11T13:05:00-04:00,PHI,None,CHC,None,PHI,None,CHC,None,-1.5,1.5,0.350,0.667,186,-200,1,PHI -1.5,Jesús Luzardo,Ben Brown,4.46,5.37,1.44,1.43,83,78,72.2,63.2,0.282,0.275,0.567,0.612,.253,.258,4.03,3.65,1.149584,1.234177,38,41,29,26,-1,1,4.6,5.6,4.4,4.0,-0.2,-0.1,0.0,1.5,3.0,-3.0,0.522388,0.656716,0.2,0.6,0.45,0.65,0.533333,0.633333,4.55,5.58,2564,2625,2283,2335,305,374,578,603,102,123,9,15,72,71,289,366,60,84,235,245,526,530,.253,.258,.327,.330,.396,.445,.724,.775,101,122,0.044678,0.052677,0.003942,0.006424,0.030223,0.040257,0.126588,0.156745,4.37,3.97,4.03,3.65,20,15,580,557,293,266,269,243,207,188,627,518,104,104,3.67,3.94,1.309,1.242,8.7,8.4
1,2025-06-11T14:10:00-04:00,ATL,None,MIL,None,MIL,None,ATL,None,-1.5,1.5,0.481,0.537,108,-116,0,MIL +1.5,Spencer Schwellenbach,Chad Patrick,3.24,2.84,1.05,1.21,75,63,80.2,69.2,0.231,0.241,0.424,0.529,.243,.235,3.76,3.83,0.935162,0.910405,28,36,38,32,-1,1,4.0,4.4,4.0,4.1,0.0,0.1,0.1,0.3,-6.0,0.0,0.515152,0.529412,0.2,0.6,0.25,0.65,0.366667,0.566667,4.05,4.37,2508,2563,2241,2261,267,297,545,532,94,91,6,5,74,71,257,274,36,86,226,240,568,558,.243,.235,.317,.314,.386,.361,.703,.675,97,92,0.041946,0.040248,0.002677,0.002211,0.031682,0.026979,0.114681,0.121185,3.95,4.07,3.76,3.83,10,16,500,542,261,277,243,256,218,245,586,557,109,104,3.98,4.09,1.233,1.307,7.7,8.1
2,2025-06-11T14:15:00-04:00,STL,None,TOR,None,TOR,None,STL,None,-1.5,1.5,0.377,0.635,165,-174,0,STL -1.5,Matthew Liberatore,Eric Lauer,3.82,2.08,1.13,0.88,58,24,68.1,26.0,0.249,0.157,0.537,0.552,.258,.256,3.96,4.09,0.851689,0.923077,36,37,31,30,-3,2,4.7,4.3,4.3,4.3,0.1,0.0,0.5,0.0,0.0,3.0,0.537313,0.507463,0.4,0.8,0.50,0.70,0.600000,0.666667,4.69,4.34,2553,2547,2278,2265,314,291,588,579,120,118,4,3,55,94,299,280,37,37,215,222,517,456,.258,.256,.328,.325,.396,.402,.724,.727,104,103,0.052678,0.052097,0.001756,0.001325,0.027217,0.030464,0.131255,0.123620,4.30,4.33,3.96,4.09,17,22,577,532,288,290,262,272,183,200,474,599,103,103,3.71,4.26,1.276,1.223,8.7,8.0
3,2025-06-11T13:10:00-04:00,CIN,None,CLE,None,CLE,None,CIN,None,-1.5,1.5,0.410,0.605,144,-153,0,CLE +1.5,Nick Lodolo,Logan Allen,3.21,4.42,1.08,1.58,64,40,75.2,55.0,0.239,0.280,0.515,0.515,.246,.232,3.66,3.92,0.851064,0.727273,35,34,33,32,5,-2,4.5,3.8,3.9,4.2,-0.1,0.1,0.5,-0.2,-4.0,4.0,0.573529,0.454545,0.6,0.4,0.55,0.45,0.533333,0.433333,4.54,3.85,2560,2419,2280,2150,309,254,561,499,114,90,7,6,76,71,297,240,59,53,238,211,601,535,.246,.232,.321,.305,.398,.373,.719,.678,95,91,0.050000,0.041860,0.003070,0.002791,0.032018,0.031163,0.130263,0.111628,3.93,4.20,3.66,3.92,19,19,509,559,267,277,245,254,204,240,541,562,121,104,4.06,4.06,1.182,1.371,7.6,8.6
4,2025-06-11T19:40:00-04:00,NYY,None,KAN,None,KAN,None,NYY,None,-1.5,1.5,0.421,0.608,138,-155,0,KAN +1

✅ Enriched training set saved to ../training-data/training-set/training_set_2025-06-11.csv
